# LLM Two-stages Fine-Tunning

---



In [ ]:
!pip install PyMuPDF
!pip install unsloth trl peft accelerate bitsandbytes
!pip install langchain langchain-community langchain-huggingface chromadb sentence-transformers
!pip install -qU langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 90.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# For GPU check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

CUDA available: True
GPU: Tesla T4


# Model Selection: Llama 3

In [ ]:
from unsloth import FastLanguageModel
import torch

#model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit"
model_name = "unsloth/Llama-3.2-1B"

max_seq_length = 2048  # Choose sequence length
dtype = None  # Auto detection

# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-1b-unsloth-bnb-4bit as a legacy tokenizer.


# 1. Domain Specific Fine-Tunning

## Load and Process PDF Data


In [ ]:
import fitz

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

In [ ]:
import re

def split_paragraphs(pages, chunk_size=50):
    # Combine all pages into one large text
    full_text = " ".join(pages)

    # 1. Preserve actual word boundaries:
    # Replace 2 or more spaces (or newlines) with a temporary placeholder
    full_text = re.sub(r'\s{2,}', '___BOUNDARY___', full_text)

    # 2. Squash the stray letters together by removing all single spaces
    full_text = full_text.replace(' ', '')

    # 3. Restore the actual spaces between words
    full_text = full_text.replace('___BOUNDARY___', ' ')

    # 4. Clean up any remaining messy whitespace
    clean_text = re.sub(r'\s+', ' ', full_text).strip()

    # Split into actual words
    words = clean_text.split()

    # Group words into chunks
    paragraphs = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i : i + chunk_size])
        paragraphs.append(chunk)

    return paragraphs

In [ ]:
import os
import glob

# 1. Define the directory where your books are stored
pdf_directory = "/content/drive/MyDrive/LLM/Books"

# 2. Find all files ending in .pdf within that directory
pdf_files = glob.glob(os.path.join(pdf_directory, "*.pdf"))

# This master list will hold the chunks from EVERY book
all_pdf_chunks = []

# 3. Loop through each PDF file found
for pdf_path in pdf_files:
    print(f"Processing: {os.path.basename(pdf_path)}...")

    # Extract text from the current PDF in the loop
    raw_text = extract_text_from_pdf(pdf_path)

    # Split the text into chunks
    chunks = split_paragraphs(raw_text)

    # Add this book's chunks to our master list
    all_pdf_chunks.extend(chunks)

# 4. Verify the final results across all books
print("-" * 30)
print(f"Successfully processed {len(pdf_files)} PDF(s).")
print(f"Total chunks combined: {len(all_pdf_chunks)}")

if all_pdf_chunks:
    print(f"First chunk overall: {all_pdf_chunks[0]}")
else:
    print("No chunks were generated. Please check if the folder contains readable PDFs.")

Processing: SV06 Leveling Guide and Tips.pdf...
Processing: 3D-Printing-Guide.pdf...
Processing: 87359ff507db60fefc79788d912edc31d13f.pdf...
Processing: d41586-020-00271-6.pdf...
Processing: ilide.info-3d-printing-for-dummies-3rd-edition-horne-pr_cb7b313d5fdb180480123f7e4f41454a.pdf...
Processing: L-0003936151-pdf.pdf...
Processing: ilide.info-3d-printing-e-book-with-content-pdf-pr_df7c7813f361a42c6c7339bc2e11ec30.pdf...
Processing: processes-13-01772.pdf...
Processing: F0Q5PYLJMV0TH4G.pdf...
Processing: 3DPrinting_eBook54.pdf...
Processing: 3637288.pdf...
Processing: ChemRxiv2023-pre-print.pdf...
Processing: tesi.pdf...
Processing: make-getting-started-with-3d-printing-2nbsped-9781680456431-1680456431_compress.pdf...
Processing: 127450633.pdf...
Processing: bell2014.pdf...
Processing: 140919-common-3d-printing-failures-wmf-2014-nyc-v02.pdf...
Processing: 3DP-2018-2.pdf...
Processing: When_3D_Prints_go_wrong_Revised2.pdf...
Processing: basics-of-3D-printing.pdf...
Processing: 3D Printi

## Create Domain-Specific Dataset


In [ ]:
from datasets import Dataset

# Create the dataset from the text chunks
domain_dataset = Dataset.from_dict({"text": all_pdf_chunks})

# Verify the dataset
print(domain_dataset)

Dataset({
    features: ['text'],
    num_rows: 28927
})


In [ ]:
print(domain_dataset[-20])

{'text': "Prototypes of a ski goggles' frame printed with FDM, SLA and SLS technology (from left to right). Source: https://formlabs.com/de/blog/fdm-vs-sla-vs-sls-how-to-choose-the- right-3d-printing-technology/ [10] Figure 6: 3D Printing Technologies Comparison. Source: STP [11] Universe Berkeley: Mechanical Engineering Student Access Machine Shop: Stratus Dimensions Fused Deposition Modelling. [12] Figure 7: FDM Technology. Source: https://i.materialise.com/blog/3d-printing-technologies-"}


In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=64,  # LoRA rank - higher = more capacity, more memory
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=128,  # LoRA scaling factor (usually 2x rank)
    lora_dropout=0,  # Supports any, but = 0 is optimized
    bias="none",     # Supports any, but = "none" is optimized
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized version
    random_state=3407,
    use_rslora=False,  # Rank stabilized LoRA
    loftq_config=None, # LoftQ
)

Unsloth 2026.5.7 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


In [ ]:
#!pip install -q trl
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

# Configure training arguments for domain pre-training
domain_training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="domain_pretrain_outputs",
    save_strategy="epoch",
    report_to="none"
)

# Initialize the SFTTrainer
domain_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=domain_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=domain_training_args,
)

Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/28927 [00:00<?, ? examples/s]

## Execute First Stage Fine-tuning

In [ ]:
from transformers.trainer_callback import ProgressCallback

# Remove the ProgressCallback to stop the notebook progress bars from flooding the output
domain_trainer.remove_callback(ProgressCallback)
domain_trainer.args.disable_tqdm = True

# Train the model on the domain dataset
domain_train_stats = domain_trainer.train()

# Print training statistics
print(domain_train_stats)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 28,927 | Num Epochs = 1 | Total steps = 3,616
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 45,088,768 of 1,280,903,168 (3.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,3.330674
20,3.057981
30,3.110506
40,3.079307
50,2.920822
60,2.950874
70,2.940395
80,3.048779
90,2.932601
100,2.922092


Unsloth: Restored added_tokens_decoder metadata in domain_pretrain_outputs/checkpoint-3616/tokenizer_config.json.


TrainOutput(global_step=3616, training_loss=2.7037680323145032, metrics={'train_runtime': 2208.9181, 'train_samples_per_second': 13.096, 'train_steps_per_second': 1.637, 'total_flos': 1.493196445538304e+16, 'train_loss': 2.7037680323145032, 'epoch': 1.0})


In [ ]:
model.save_pretrained("domain_specific_lora")
tokenizer.save_pretrained("domain_specific_lora")

model.save_pretrained_merged("domain_merged_model", tokenizer, save_method="merged_16bit")

Unsloth: Restored added_tokens_decoder metadata in domain_specific_lora/tokenizer_config.json.


config.json:   0%|          | 0.00/889 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in domain_merged_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:07<00:00,  7.89s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:08<00:00,  8.05s/it]


Unsloth: Merge process complete. Saved to `/content/domain_merged_model`


# 2. RAG Mechanism

In [ ]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

drive.mount('/content/drive')
rag_directory = "/content/drive/MyDrive/LLM/RAG books"


# Initialize the document loader specifically targeting PDF formats
# PyMuPDFLoader is utilized for accurate extraction of complex formatting and tables
print("Initializing document ingestion...")
loader = DirectoryLoader(
    rag_directory,
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True
)
documents = loader.load()
print(f"Successfully loaded {len(documents)} document pages from '{rag_directory}'.")

/tmp/ipykernel_1286/2979293849.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Initializing document ingestion...


 17%|█▋        | 5/30 [00:02<00:15,  1.64it/s]

MuPDF error: syntax error: cannot find ExtGState resource 'R32'

MuPDF error: syntax error: cannot find XObject resource 'R86'

MuPDF error: syntax error: cannot find ExtGState resource 'R32'

MuPDF error: syntax error: cannot find XObject resource 'R93'

MuPDF error: syntax error: cannot find ExtGState resource 'R32'

MuPDF error: syntax error: cannot find XObject resource 'R121'

MuPDF error: syntax error: cannot find ExtGState resource 'R32'

MuPDF error: syntax error: cannot find XObject resource 'R128'

MuPDF error: syntax error: cannot find ExtGState resource 'R32'

MuPDF error: syntax error: cannot find XObject resource 'R151'

MuPDF error: syntax error: cannot find ExtGState resource 'R32'

MuPDF error: syntax error: cannot find XObject resource 'R157'

MuPDF error: syntax error: cannot find ExtGState resource 'R32'

MuPDF error: syntax error: cannot find XObject resource 'R163'

MuPDF error: syntax error: cannot find ExtGState resource 'R32'

MuPDF error: syntax error: cannot 

100%|██████████| 30/30 [00:31<00:00,  1.04s/it]

Successfully loaded 1996 document pages from '/content/drive/MyDrive/LLM/RAG books'.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ==========================================
# UPGRADED RAG CHUNKING CONFIGURATION
# ==========================================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=300,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

print("Segmenting documents into semantic chunks...")
chunks = text_splitter.split_documents(documents)
print(f"Generated {len(chunks)} independent text chunks.")

Segmenting documents into semantic chunks...
Generated 2523 independent text chunks.


In [ ]:
# Initialize the embedding model
# The 'all-MiniLM-L6-v2' model offers an optimal balance between semantic accuracy
# and computational efficiency within a local notebook environment.
print("Initializing embedding model...")
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda'}, # Leverage GPU acceleration if available
    encode_kwargs={'normalize_embeddings': True}
)

# Construct and persist the Chroma vector database
vector_store_path = "./chroma_fdm_db"
print(f"Constructing vector database at '{vector_store_path}'...")

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=vector_store_path
)

print("RAG vector database successfully generated and persisted to disk.")

Initializing embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Constructing vector database at './chroma_fdm_db'...
RAG vector database successfully generated and persisted to disk.


In [ ]:
!zip -r chroma_fdm_db.zip ./chroma_fdm_db

  adding: chroma_fdm_db/ (stored 0%)
  adding: chroma_fdm_db/99958a98-4e09-46b0-82ff-110ded6391a7/ (stored 0%)
  adding: chroma_fdm_db/99958a98-4e09-46b0-82ff-110ded6391a7/link_lists.bin (deflated 79%)
  adding: chroma_fdm_db/99958a98-4e09-46b0-82ff-110ded6391a7/index_metadata.pickle (deflated 45%)
  adding: chroma_fdm_db/99958a98-4e09-46b0-82ff-110ded6391a7/length.bin (deflated 82%)
  adding: chroma_fdm_db/99958a98-4e09-46b0-82ff-110ded6391a7/header.bin (deflated 58%)
  adding: chroma_fdm_db/99958a98-4e09-46b0-82ff-110ded6391a7/data_level0.bin (deflated 11%)
  adding: chroma_fdm_db/chroma.sqlite3 (deflated 56%)


In [ ]:
from google.colab import files
print("Zipping complete. Initiating download...")
files.download('chroma_fdm_db.zip')

Zipping complete. Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## RAFT Dataset

In [ ]:
import json
import random

# 1. Configuration
input_filename = "FinalDataset.json"
output_filename = "RAFT_FinalDataset.json"

# Load the original dataset
with open(input_filename, "r", encoding="utf-8") as f:
    dataset = json.load(f)

# Pre-extract all questions to use for the "Distractor" RAG searches
all_questions = []
for item in dataset:
    user_content = item["messages"][0]["content"]
    # Extract just the question (everything before the first double line break)
    question = user_content.split("\n\n")[0]
    all_questions.append(question)

new_dataset = []

print(f"Processing {len(dataset)} samples for RAFT distribution...")

# 2. Process the Dataset
for i, item in enumerate(dataset):
    # Deep copy to avoid modifying the original list in place
    new_item = json.loads(json.dumps(item))

    user_content = new_item["messages"][0]["content"]

    # Split the user content into the Question and the Telemetry Data
    parts = user_content.split("\n\n", 1)
    actual_question = parts[0]
    telemetry_data = parts[1] if len(parts) > 1 else ""

    # Generate a random float between 0.0 and 1.0 to decide the RAG injection type
    roll = random.random()

    if roll < 0.75:
        # ---------------------------------------------------------
        # GOLDEN RAG (50% of data): Query the DB with the actual question
        # ---------------------------------------------------------
        retriever = vector_db.as_retriever(search_kwargs={"k": 1})
        docs = retriever.invoke(actual_question)

        # Clean up newlines so the context fits neatly on one line
        context_text = " ".join([doc.page_content for doc in docs]).replace("\n", " ")

        # Rebuild the prompt
        new_user_content = f"{actual_question} (Reference: {context_text})\n\n{telemetry_data}"


    else:
        # ---------------------------------------------------------
        # ZERO RAG (25% of data): Leave the prompt exactly as it is
        # ---------------------------------------------------------
        new_user_content = user_content

    # Update the item with the new constructed user content
    new_item["messages"][0]["content"] = new_user_content
    new_dataset.append(new_item)

# 3. Save the new RAFT dataset
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(new_dataset, f, indent=4)

print(f"Successfully generated {output_filename}!")
print("RAFT Distribution applied: ~50% Golden, ~25% Distractor, ~25% Zero")

Processing 500 samples for RAFT distribution...
Successfully generated RAFT_FinalDataset.json!
RAFT Distribution applied: ~50% Golden, ~25% Distractor, ~25% Zero


In [ ]:
import json

# Define file paths
input_filename = "RAFT_FinalDataset.json"
output_filename = "RAFT_Ready_For_API.json"

# Load the dataset
with open(input_filename, "r", encoding="utf-8") as f:
    dataset = json.load(f)

cleared_count = 0
untouched_count = 0

# Process the dataset
for item in dataset:
    messages = item.get("messages", [])

    # Ensure the standard user-assistant structure exists
    if len(messages) >= 2 and messages[0]["role"] == "user" and messages[1]["role"] == "assistant":
        user_prompt = messages[0]["content"]

        # Check if this sample received a RAG injection
        if "(Reference:" in user_prompt:
            # Clear the assistant's response
            messages[1]["content"] = ""
            cleared_count += 1
        else:
            untouched_count += 1

# Save the prepared dataset
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=4)

# Output the processing metrics
print("--- Dataset Preparation Complete ---")
print(f"Total samples processed: {len(dataset)}")
print(f"Assistant responses cleared (Golden/Distractor RAG): {cleared_count}")
print(f"Assistant responses preserved (Zero RAG): {untouched_count}")
print(f"Output saved to: {output_filename}")

--- Dataset Preparation Complete ---
Total samples processed: 500
Assistant responses cleared (Golden/Distractor RAG): 361
Assistant responses preserved (Zero RAG): 139
Output saved to: RAFT_Ready_For_API.json


In [ ]:
import json

# Define file paths using the output from the previous step
input_filename = "RAFT_Ready_For_API.json"
output_filename = "RAFT_Sorted_For_API.json"

# Load the dataset
with open(input_filename, "r", encoding="utf-8") as f:
    dataset = json.load(f)

cleared_samples = []
untouched_samples = []

# Sort the dataset into two separate lists
for item in dataset:
    messages = item.get("messages", [])

    if len(messages) >= 2 and messages[1]["role"] == "assistant":
        # If the assistant content is strictly empty, it goes to the cleared list
        if messages[1]["content"] == "":
            cleared_samples.append(item)
        else:
            untouched_samples.append(item)

# Combine the lists: Cleared samples first, followed by the untouched ones
sorted_dataset = cleared_samples + untouched_samples

# Save the newly sorted dataset
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(sorted_dataset, f, indent=4)

print("--- Sorting Complete ---")
print(f"Blank samples moved to the top: {len(cleared_samples)}")
print(f"Completed samples pushed to the bottom: {len(untouched_samples)}")
print(f"Output saved to: {output_filename}")

--- Sorting Complete ---
Blank samples moved to the top: 361
Completed samples pushed to the bottom: 139
Output saved to: RAFT_Sorted_For_API.json


# Second Stage: Instructional Fine-Tunning



In [ ]:
import json
from datasets import Dataset
# 1. Load raw data
file = json.load(open("RAFT_FinalDataset.json"))
print(f"Number of samples: {len(file)}")

# 2. Convert to HF Dataset
dataset = Dataset.from_list(file)


Number of samples: 500


In [ ]:
from unsloth import FastLanguageModel

# ==========================================
# PHASE 1: LOAD MODEL & TOKENIZER
# ==========================================
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="domain_merged_model",
    #model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# ==========================================
# PHASE 2: CONFIGURE LLAMA 3 TEMPLATE & TOKENS
# ==========================================
llama3_chat_template = (
    "{{ '<|begin_of_text|>' }}"
    "{% for message in messages %}"
    "{% if message['role'] == 'user' %}"
    "{{ '<|start_header_id|>user<|end_header_id|>\n\n' + message['content'] + '<|eot_id|>' }}"
    "{% elif message['role'] == 'assistant' %}"
    "{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' + message['content'] + '<|eot_id|>' }}"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}"
    "{% endif %}"
)

tokenizer.chat_template = llama3_chat_template

# CRITICAL: Set the padding and end tokens so the model learns the hard stop
tokenizer.eos_token = "<|eot_id|>"
tokenizer.pad_token = "<|eot_id|>"
tokenizer.padding_side = "right"

# ==========================================
# PHASE 3: FORMAT AND MAP THE DATASET
# ==========================================
def formatting_prompts_func(examples):
    convos = examples["messages"]
    # The tokenizer is now loaded and properly configured with the template
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched=True)

# Verify the output
print("--- Format Verification ---")
print(dataset[1]['text'])

==((====))==  Unsloth 2026.5.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

The tokenizer you are loading from 'domain_merged_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from 'domain_merged_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load domain_merged_model as a legacy tokenizer.


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

--- Format Verification ---
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

How does increasing retraction distance affect stringing on direct drive extruders? (Reference: tip. To ensure retraction is enabled, click “Edit Process Settings” and click on the  Extruder tab. Ensure that the retraction option is enabled for each of your extruders. In  the sections below, we will discuss the important retraction settings as well as several  other settings that can be used to combat stringing, such as the extruder temperature  settings.  Common Solutions  Retraction distance  The most important retraction setting is the retraction distance. This determines how  much plastic is pulled out of the nozzle. In general, the more plastic that is retracted  from the nozzle, the less likely the nozzle is to ooze while moving. Most direct-drive  extruders only require a retraction distance of 0.5-2.0mm, while some Bowden  extruders may require a retraction distance as high as 15mm due to the

In [ ]:
# ==========================================
# 1. ATTACH LORA ADAPTERS (The "Brain Surgery")
# ==========================================
# This must execute immediately before the SFTTrainer
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch


# ==========================================
# 2. INITIALIZE THE TRAINER
# ==========================================
trainer = SFTTrainer(
    model=model, # The model now explicitly contains the adapters
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=30,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="epoch",
        save_total_limit=2,
        dataloader_pin_memory=False,
        report_to="none",
    ),
)

# Execute the training
trainer_stats = trainer.train()

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 63
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
10,2.545848
20,2.145297
30,1.839296
40,1.583405
50,1.521626
60,1.516551


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-63/tokenizer_config.json.


In [ ]:
import torch
from unsloth import FastLanguageModel

# 1. Prepare model for inference
FastLanguageModel.for_inference(model)

# 2. Define the input data
instruction = "What is happening with my printer? Be specific"
input_data = """Process parameters:
Print time in minutes: 10.45
Nozzle temperature: 212.50
Bed temperature: 60.00
X acceleration: 1.20
Y acceleration: -2.00
Z acceleration: 0.10
Amb temperature: 24.00
Amb humidity: 45.00
Current of the X axis motor: 0.90
Current of the Y axis motor: 0.90
Current of the Z axis motor: 0.85
Current of the filament motor: 1.10
Filament: 1"""

# 3. Format the prompt
messages = [{"role": "user", "content": f"{instruction}\n\n{input_data}"}]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

# 4. Define precise stopping conditions
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>"),
    tokenizer.convert_tokens_to_ids("<|end_of_text|>")
]

# 5. Generate with Sampling Enabled
outputs = model.generate(
    input_ids=inputs,
    attention_mask=torch.ones_like(inputs),
    max_new_tokens=200,
    use_cache=True,

    # --- ADD THESE TWO LINES FOR VARIETY ---
    do_sample=True,    # This tells the model to roll the dice on word choice
    temperature=0.6,  # Slightly higher so it has room to pick different words

    repetition_penalty=1.05,
    eos_token_id=terminators,
    pad_token_id=tokenizer.eos_token_id,
)

# 6. Decode and Sanitize
input_length = inputs.shape[1]
# We set skip_special_tokens=False so our sanitize function can detect the tags
raw_response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=False)

def sanitize_output(text):
    # This targets the exact prefix of the hallucinated tags and chops everything after it
    if "<|reserved" in text:
        text = text.split("<|reserved")[0]
    if "<|eot_id" in text:
        text = text.split("<|eot_id")[0]
    if "<|start_header" in text:
        text = text.split("<|start_header")[0]
    if "assistant" in text:
        text = text.split("assistant")[0]

    return text.strip()

final_clean_response = sanitize_output(raw_response)

print("--- FINAL CLEAN RESPONSE ---")
print(final_clean_response)

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

--- FINAL CLEAN RESPONSE ---
The machine is currently in the Active Printing phase. The bed is currently at 60.00 °C, so the plastic is melting and oozing out of the nozzle. Unfortunately, your filament sensor is reading a very high 1.10 A/C, which means the motor is drawing an extremely heavy 1.20 g of current to move those heavy metal spools around. If you keep this up for long enough, it will overheat the motor and cause a thermal runaway. Please click "Reduce Print Speed" in the upper right corner to lower the printing speed.


## Download Model

In [ ]:
from google.colab import userdata

# Get your Hugging Face token from Colab secrets.
# If you don't have one, please create it in your Hugging Face settings
# and then add it to Colab secrets under the name 'HF_TOKEN'.
hf_token = "****"


In [ ]:
import gc
import torch

# 1. Delete the trainer to free up massive amounts of RAM
if 'trainer' in globals():
    del trainer

# 2. Force Python to run garbage collection
gc.collect()

# 3. Clear the GPU cache
torch.cuda.empty_cache()

# 4. Now execute the local GGUF conversion
# (q4_k_m is an excellent balance of speed, memory, and retention of FDM knowledge)
print("Starting GGUF conversion... Do not interrupt the notebook.")
model.save_pretrained_gguf(
    "local_gguf_model2",
    tokenizer,
    quantization_method="q4_k_m"
)
print("Conversion complete!")



Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Starting GGUF conversion... Do not interrupt the notebook.
Unsloth: Merging model weights to 16-bit format...
Detected local model directory: /content/domain_merged_model


Unsloth: Restored added_tokens_decoder metadata in local_gguf_model2/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:04<00:00,  4.46s/it]


Copied model.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:09<00:00,  9.98s/it]


Unsloth: Merge process complete. Saved to `/content/local_gguf_model2`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['local_gguf_model2_gguf/domain_merged_model.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: M

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: All GGUF conversions completed successfully!
Generated files: ['local_gguf_model2_gguf/domain_merged_model.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'domain_merged_model'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model local_gguf_model2_gguf/domain_merged_model.Q4_K_M.gguf -p "why is the sky blue?"
Conversion complete!


In [ ]:
import os
from google.colab import drive

# 2. Define your target Drive path
drive_path = "/content/drive/MyDrive/LLM models/Llama 3.1 final/gguf_model/"

# 3. Safely create the destination folder if it doesn't exist yet
os.makedirs(drive_path, exist_ok=True)

# 4. Execute the copy commands (without hiding errors)
print("Copying files to Google Drive. This may take a few minutes for a 5GB file...")

# Copy from the folder we created in the previous step
!cp -r local_gguf_model2_gguf/* "{drive_path}"

# Fallback: Copy any stray .gguf files or Modelfiles in the root directory
!cp *.gguf "{drive_path}"
!cp Modelfile "{drive_path}"

print(f"Transfer complete! Please verify your files are visible at: {drive_path}")

Copying files to Google Drive. This may take a few minutes for a 5GB file...
cp: cannot stat '*.gguf': No such file or directory
cp: cannot stat 'Modelfile': No such file or directory
Transfer complete! Please verify your files are visible at: /content/drive/MyDrive/LLM models/Llama 3.1 final/gguf_model/


In [ ]:
import os
from google.colab import files


# 1. Define the folder path and the name of the zip file you want to create
folder_path = "/content/drive/MyDrive/LLM models/Llama 3.1 final/gguf_model"
zip_name = "llama3_model_folder.zip"
print(os.path.exists(folder_path))
# 2. Create the zip archive
# This command zips the folder at 'folder_path' into 'zip_name'
os.system(f'zip -r "{zip_name}" "{folder_path}"')

# 3. Download the zipped file
files.download(zip_name)


True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!cp "/content/llama3_model_folder.zip" "/content/drive/MyDrive/LLM models/Llama 3 final"

In [ ]:
drive.flush_and_unmount()

# TESTING RAG-MODEL

In [ ]:
import torch
from unsloth import FastLanguageModel

# Enable native 2x faster inference for the Unsloth model
FastLanguageModel.for_inference(model)

def test_rag_inference(question: str, printer_data: str) -> None:
    """
    Evaluates the RAG pipeline by retrieving semantic context and
    generating an informed response using the fine-tuned model.
    """
    print(f"Executing semantic search for query: '{question}'...")

    # 1. Retrieve relevant context from the local Chroma database
    retriever = vector_db.as_retriever(search_kwargs={"k": 1})
    retrieved_docs = retriever.invoke(question)

    # Consolidate the extracted paragraphs
    context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])
    print(f"Successfully retrieved {len(retrieved_docs)} document chunks.\n")
    print("=== CONTEXT ===")
    print(context_text)
    print("==============\n")

    # 2. Construct the message utilizing ONLY the user role.
    # We inject the strict persona, the RAG context, the question, and the telemetry block here.
    augmented_prompt = (
        f"{question} (Reference: )\n"
        #f"{question} (Reference: {context_text})\n\n"
        f"{printer_data}"
    )

    messages = [
        {"role": "user", "content": augmented_prompt}
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    # ==========================================
    # 2. THE GENERATION FIXES
    # ==========================================
    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>"),
        tokenizer.convert_tokens_to_ids("<|end_of_text|>")
    ]

    print("Generating educational response...")
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=torch.ones_like(input_ids),
        max_new_tokens=512,
        use_cache=True,

        # --- CRITICAL FIXES HERE ---
        do_sample=True,          # Turn sampling back on to break loops
        temperature=0.01,        # Keep it almost zero so it remains highly factual
        repetition_penalty=0.99,  # 1.0 means OFF. This stops the Chinese characters!

        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id,
    )

    # 3. Extract, Decode, and Sanitize
    input_length = input_ids.shape[1]
    raw_response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=False)

    def sanitize_output(text):
        if "<|reserved" in text: text = text.split("<|reserved")[0]
        if "<|eot_id" in text: text = text.split("<|eot_id")[0]
        if "<|start_header" in text: text = text.split("<|start_header")[0]
        if "assistant" in text: text = text.split("assistant")[0]
        return text.strip()

    final_clean_response = sanitize_output(raw_response)

    print("\n=== LLM RESPONSE ===")
    print(final_clean_response)
    print("====================")

# ==========================================
# TEST EXECUTION
# ==========================================
test_question = "What is 3D printing?"

# Replaced the flat array with the strictly formatted parameter block
test_sensor_array = """Process parameters:
Print time in minutes: 12.50
Nozzle temperature: 215.00
Bed temperature: 60.00
X acceleration: 1.50
Y acceleration: -1.20
Z acceleration: 0.00
Amb temperature: 23.50
Amb humidity: 42.00
Current of the X axis motor: 0.90
Current of the Y axis motor: 0.90
Current of the Z axis motor: 0.85
Current of the filament motor: 1.10
Filament: 1"""

test_rag_inference(test_question, test_sensor_array)

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Executing semantic search for query: 'What is 3D printing?'...
Successfully retrieved 1 document chunks.

=== CONTEXT ===
5
WHAT IS 3D PRINTING?

Generating educational response...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



=== LLM RESPONSE ===
Hello! We are in the Active Printing phase. The filament motor is drawing a very high 1.10 A, indicating a very fast, heavy print. The bed is at a very cold 60.00 °C, so the plastic is literally freezing. The nozzle is at a very hot 215.00 °C, so the plastic is literally melting! The printer is working incredibly hard to keep the bed warm. It is currently drawing a massive 1.50 A of electrical current to keep the bed warm. The bed is at a very cold 60.00 °C, so the plastic is literally freezing! The nozzle is at a very hot 215.00 °C, so the plastic is literally melting! The printer is working incredibly hard to keep the bed warm. It is currently drawing a massive 1.50 A of electrical current to keep the bed warm. The bed is at a very cold 60.00 °C, so the plastic is literally freezing! The nozzle is at a very hot 215.00 °C, so the plastic is literally melting! The printer is working incredibly hard to keep the bed warm. It is currently drawing a massive 1.50 A of 